# Navie RAG

쉽게 말해 이게 LLM의 가장 기본적인 RAG  파이프라인이다
나이브 RAG의 9단계

1. 문서 로드
2. 문서 분할
3. 임베딩
4. 벡터스토어
5. 리트리벌
6. 프롬프트
7. 체이닝 
8. 질문
9. 답변

이렇게가 나이브 RAG 즉, 가장 기본적인 랭체인을 이용한 RAG의 파이프라인이다. 

## RAG 단계 한 번에 살펴보기

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

import os
import time
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

os.environ["GOOGLE_API_KEY"] = api_key

In [ ]:
# 단계 1: 문서 로드(Load Documents)
loader = PyMuPDFLoader("/home/pck/메타코드 부트캠프/13주차/01.Basic-20260501T061411Z-3-001/01.Basic/data/SPRi AI Brief_6월호_산업동향_F.pdf")
# loader = PDFPlumberLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

In [ ]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 일반적으로 overlap을 해주는 것이 조금 더 좋다. 청크사이즈에 따라서 문맥이 짤리는 것을 방지학 위해서 오버랩을 준다. 청크 사이즈의 일반적으로 10~20% 정도 오버랩 한다(강사님은)

In [ ]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT"
)

In [ ]:
# 단계 4: DB 생성(Create DB) 및 저장
# 문서는 RETRIEVAL_DOCUMENT 방식으로 임베딩되어 벡터스토어에 저장됩니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)
print(f"FAISS 인덱스 생성 완료: documents={len(split_documents)}")


In [ ]:
# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

In [ ]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt_template = PromptTemplate.from_template(
    
    """You are an helpful assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Answer in Korean.

## INSTRUCTIONS:
   - QUESTION에 대해서 Context 를 기반으로 답변을 작성해줘.
   - 만약 Context 에 답이 없다면 "모르겠습니다" 라고 대답해줘.
   
## ANSWER FORMAT:
    - 항상 인사를 하고 답변을 시작해
    - 두괄식으로 질문에 대해서 전략적인 답변을 먼저 해주고, 그다음 2~3 줄 정도로 답변에 대한 근거를 작성해줘
    - Use bullet points for lists.
    - Keep your answers concise and to the point.


답변할 때는 단계 별로 생각해서 답변해줘 (Answering step-by-step)


#Context: 
{context}

#Question:
{question}

#Answer:
"""
)

## 여기 템플릿 안에는 마크다운 같이 생겨야 한다. 즉, ##은 주석 처리가 아닌거지!
# #Answer:  여기부터 답변을 작성하라고 알려주는 역할

In [ ]:
# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()} 
    | prompt_template
    | llm
    | StrOutputParser() # 스트링으로 output이 나오도록 하는 아웃풋 파서이다. 단순히 LLM 답변이 아니라 이것까지 추가하면 LLM 답변이 조금 더 완결성 있다. 
)

# RunnablePassthrough-> 질문 자체가 변하지 않고 변형되지 않고 체이닝 통해서 LLM 까지 갈 수 있게끔 하는 함수이다. 입력 받은 데이터를 바꾸거나 수정하지 않고 그 다음 구성요소로 넘기는 방식이다. 복잡한 체이닝에서 데이터를 그대로 유지하도록 하는 필수적인 요소이다(랭체인에서)


In [ ]:
# 단계 8. 질문하기

question = "현재 AI 분야에서 가장 집중해야할 이슈 하나만 말해줘"
response = chain.invoke(question)

print(response)

### 단계 9. 답변



```python
안녕하세요!

현재 AI 분야에서 가장 집중해야 할 이슈 중 하나는 **AI에 의한 노동력 교란**입니다.

이는 유능한 AI 시스템이 여러 산업 분야에 도입되어 노동력을 대체하며 기업들이 혜택을 누리는 동시에 대중의 강한 반발을 초래하고 있기 때문입니다.
```
